# EMT 3-phase mixed V-Type and I-Type SSN example

This notebook compares:

1. **Reference MNA case** with standard EMT components:
   - `VoltageSource`
   - `RBranch`
   - `CBranch`
   - `RLoad`
   - `LLoad`

2. **Mixed SSN case**:
   - `CBranch` is modeled as a **GenericTwoTerminalITypeSSN**
   - `LLoad` is modeled as a **GenericTwoTerminalVTypeSSN**
   - source and resistors remain standard EMT components

The topology is kept equivalent so that the logged signals can be compared directly.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import dpsimpy

emt = dpsimpy.emt
ph3 = emt.ph3

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

## Parameters

In [ ]:
time_step = 1e-4
final_time = 0.2
frequency = 50.0

source_voltage = 1.0 * np.exp(1j * 0.0)

i3 = np.eye(3)

branch_resistance = 0.2 * i3
branch_capacitance = 500e-6 * i3

load_resistance = 1.0 * i3
load_inductance = 0.05 * i3

## Helper functions

In [ ]:
def abc(phasor):
    return np.array(
        [
            [phasor],
            [phasor * np.exp(-1j * 2 * np.pi / 3)],
            [phasor * np.exp(+1j * 2 * np.pi / 3)],
        ],
        dtype=np.complex128,
    )


def create_capacitor_itype_ssn_matrices():
    inv_c = np.linalg.inv(branch_capacitance)

    # I-type capacitor:
    #
    # x = vC_abc
    # u = iC_abc
    # y = vC_abc
    #
    # dvC/dt = C^{-1} iC
    # vC     = x

    a = np.zeros((3, 3))
    b = inv_c
    c = i3.copy()
    d = np.zeros((3, 3))

    return a, b, c, d


def create_inductor_vtype_ssn_matrices():
    inv_l = np.linalg.inv(load_inductance)

    # V-type inductor:
    #
    # x = iL_abc
    # u = vL_abc
    # y = iL_abc
    #
    # diL/dt = L^{-1} vL
    # iL     = x

    a = np.zeros((3, 3))
    b = inv_l
    c = i3.copy()
    d = np.zeros((3, 3))

    return a, b, c, d


def load_log(sim_name):
    path = Path("logs") / sim_name / f"{sim_name}.csv"
    df = pd.read_csv(path, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    return df


def sig(df, name):
    cols = [c for c in df.columns if c == name or c.startswith(name + "_")]
    return df[cols].to_numpy()

## Reference simulation with standard MNA components

In [ ]:
def run_mna_case():
    sim_name = "EMT_Ph3_MNA_C_and_L_Reference"

    n1 = emt.SimNode("n1", PhaseType.ABC)
    n2 = emt.SimNode("n2", PhaseType.ABC)
    n3 = emt.SimNode("n3", PhaseType.ABC)
    n4 = emt.SimNode("n4", PhaseType.ABC)

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    r_branch = ph3.Resistor("RBranch")
    r_branch.set_parameters(branch_resistance)

    c_branch = ph3.Capacitor("CBranch")
    c_branch.set_parameters(branch_capacitance)

    r_load = ph3.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph3.Inductor("LLoad")
    l_load.set_parameters(load_inductance)

    vs.connect([emt.SimNode.gnd, n1])
    r_branch.connect([n1, n2])
    c_branch.connect([n2, n3])
    r_load.connect([n3, n4])
    l_load.connect([n4, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n2, n3, n4],
        [vs, r_branch, c_branch, r_load, l_load],
    )

    Logger.set_log_dir("logs/" + sim_name)
    logger = Logger(sim_name)

    logger.log_attribute("v3", n3.attr("v"))
    logger.log_attribute("i_branch", r_branch.attr("i_intf"))
    logger.log_attribute("v_cap", c_branch.attr("v_intf"))
    logger.log_attribute("i_cap", c_branch.attr("i_intf"))
    logger.log_attribute("i_load", r_load.attr("i_intf"))
    logger.log_attribute("v_ind", l_load.attr("v_intf"))
    logger.log_attribute("i_ind", l_load.attr("i_intf"))

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

## Mixed SSN simulation: capacitor I-type, inductor V-type

In [ ]:
def run_mixed_ssn_case():
    sim_name = "EMT_Ph3_Mixed_C_IType_and_L_VType_SSN"

    n1 = emt.SimNode("n1", PhaseType.ABC)
    n2 = emt.SimNode("n2", PhaseType.ABC)
    n3 = emt.SimNode("n3", PhaseType.ABC)
    n4 = emt.SimNode("n4", PhaseType.ABC)

    a_c, b_c, c_c, d_c = create_capacitor_itype_ssn_matrices()
    a_l, b_l, c_l, d_l = create_inductor_vtype_ssn_matrices()

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    r_branch = ph3.Resistor("RBranch")
    r_branch.set_parameters(branch_resistance)

    c_branch = ph3.GenericTwoTerminalITypeSSN("CBranchITypeSSN")
    c_branch.set_parameters(a_c, b_c, c_c, d_c)

    r_load = ph3.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph3.GenericTwoTerminalVTypeSSN("LLoadVTypeSSN")
    l_load.set_parameters(a_l, b_l, c_l, d_l)

    vs.connect([emt.SimNode.gnd, n1])
    r_branch.connect([n1, n2])
    c_branch.connect([n2, n3])
    r_load.connect([n3, n4])
    l_load.connect([n4, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n2, n3, n4],
        [vs, r_branch, c_branch, r_load, l_load],
    )

    Logger.set_log_dir("logs/" + sim_name)
    logger = Logger(sim_name)

    logger.log_attribute("v3", n3.attr("v"))
    logger.log_attribute("i_branch", r_branch.attr("i_intf"))
    logger.log_attribute("v_cap", c_branch.attr("v_intf"))
    logger.log_attribute("i_cap", c_branch.attr("i_intf"))
    logger.log_attribute("i_load", r_load.attr("i_intf"))
    logger.log_attribute("v_ind", l_load.attr("v_intf"))
    logger.log_attribute("i_ind", l_load.attr("i_intf"))

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

## Run simulations

In [ ]:
run_mna_case()
run_mixed_ssn_case()

## Load CSV logs

In [ ]:
ref_name = "EMT_Ph3_MNA_C_and_L_Reference"
mixed_name = "EMT_Ph3_Mixed_C_IType_and_L_VType_SSN"

ref = load_log(ref_name)
mixed = load_log(mixed_name)

print("Reference columns:")
print(ref.columns.tolist())

print("\nMixed SSN columns:")
print(mixed.columns.tolist())

t_ref = ref.iloc[:, 0].to_numpy()
t_mixed = mixed.iloc[:, 0].to_numpy()

np.testing.assert_allclose(t_ref, t_mixed)

## Error summary

In [ ]:
signals = [
    "v3",
    "i_branch",
    "v_cap",
    "i_cap",
    "i_load",
    "v_ind",
    "i_ind",
]

rows = []

for name in signals:
    ref_sig = sig(ref, name)
    mixed_sig = sig(mixed, name)

    assert ref_sig.shape == mixed_sig.shape, (
        name,
        ref_sig.shape,
        mixed_sig.shape,
    )

    err = mixed_sig - ref_sig

    for phase in range(ref_sig.shape[1]):
        max_abs_ref = np.max(np.abs(ref_sig[:, phase]))
        max_abs_err = np.max(np.abs(err[:, phase]))

        rows.append(
            {
                "signal": name,
                "phase": phase,
                "max_abs_error": max_abs_err,
                "rms_error": np.sqrt(np.mean(err[:, phase] ** 2)),
                "max_abs_reference": max_abs_ref,
                "relative_max_error": max_abs_err / max(max_abs_ref, 1e-12),
            }
        )

summary = pd.DataFrame(rows)
summary

## Phase A comparison plots

In [ ]:
phase = 0

for name in signals:
    ref_sig = sig(ref, name)
    mixed_sig = sig(mixed, name)

    plt.figure(figsize=(10, 5))
    plt.plot(t_ref, ref_sig[:, phase], label=f"Reference {name}")
    plt.plot(t_mixed, mixed_sig[:, phase], "--", label=f"Mixed SSN {name}")

    plt.xlabel("time [s]")
    plt.ylabel(name)
    plt.title(f"{name}, phase {phase}: reference vs mixed SSN")
    plt.grid(True)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

## Error plots

In [ ]:
for name in signals:
    ref_sig = sig(ref, name)
    mixed_sig = sig(mixed, name)
    err = mixed_sig - ref_sig

    plt.figure(figsize=(10, 5))

    for phase in range(err.shape[1]):
        plt.plot(t_ref, err[:, phase], label=f"phase {phase}")

    plt.xlabel("time [s]")
    plt.ylabel("Mixed SSN - reference")
    plt.title(f"Error: {name}")
    plt.grid(True)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

## Optional strict assertions

In [ ]:
# Adjust tolerances depending on the numerical method and the initialization.
atol = 1e-8
rtol = 1e-5

for name in signals:
    np.testing.assert_allclose(
        sig(ref, name),
        sig(mixed, name),
        atol=atol,
        rtol=rtol,
        err_msg=f"Mismatch in {name}",
    )

print("All strict assertions passed.")